# ⚠️ Cost Warning

This notebook provisions and uses **two SageMaker real-time endpoints** that incur charges as long as they are running:

- **Quantized endpoint** (`ml.g5.xlarge`): ~`$1.41`/hr
- **Full-precision endpoint** (`ml.g5.12xlarge`): ~`$7.09`/hr

**Total: ~`$8.50`/hr while both endpoints are active.**

Run the **Cleanup** cell at the bottom of this notebook or run `terraform destroy` when you are finished to stop incurring costs.

# Quantized Model Comparison for Amazon SageMaker with Qwen3-VL

This notebook supplements the blog post: **[Quantization and Deploying Models on AWS](TODO_BLOG_URL_PENDING_PUBLICATION)**.

It tests whether Unsloth's Dynamic 2.0 quantization preserves the vision-language capabilities of Qwen3-VL-8B-Instruct when compressed from 16-bit (BF16) to 4-bit (Q4_K_M GGUF). We send identical image prompts to two SageMaker endpoints — one serving the quantized model via llama.cpp, the other serving the full-precision model via vLLM — and compare the results across four dimensions:

1. **Output quality**: Do both models identify the same objects, read the same text, and describe the same scenes?
2. **Latency**: How long does each endpoint take to respond?
3. **Throughput**: How many tokens per second does each model generate?
4. **Cost**: What is the per-request cost difference (`$1.41/hr` vs `$7.09/hr`)?

The goal is to determine whether the 5x cost savings from quantization comes with an acceptable trade-off in output quality for practical image understanding tasks.

**Note on prompt formatting:** Both endpoints use the same chat completion request format, so prompt formatting is controlled across the comparison. This matters because differences in chat templates, EOS handling, or prompt structure can cause a good model to appear worse than it is — an issue the companion blog discusses in detail.

## Architecture

![Quantized Model Comparison Architecture](https://raw.githubusercontent.com/aws-samples/sample-quantized-ML-model-comparison/main/images/architecture-diagram.png)

In [ ]:
import json
import time
import base64
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image as IPImage

from comparison_utils import (
    InferenceResult,
    ComparisonResult,
    PRICING,
    get_pricing,
    calculate_latency,
    calculate_throughput,
    calculate_cost_per_request,
    compute_average_metrics,
    compute_grouped_averages,
    encode_image,
    build_quantized_payload,
    build_full_precision_payload,
    invoke_endpoint,
    run_comparison,
)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Update these endpoint names to match your Terraform outputs
QUANTIZED_ENDPOINT = "qwen3-vl-8b-quantized"
FULL_PRECISION_ENDPOINT = "qwen3-vl-8b-full-precision"
AWS_REGION = "us-east-2"

# Generation parameters
GENERATION_PARAMS = {
    "max_tokens": 200,
    "temperature": 0.1,
}

# Config dict for run_comparison()
CONFIG = {
    "quantized_endpoint": QUANTIZED_ENDPOINT,
    "full_precision_endpoint": FULL_PRECISION_ENDPOINT,
    "aws_region": AWS_REGION,
}

# Metrics store — accumulates results across all prompts
metrics_store: list[ComparisonResult] = []

print(f"Quantized endpoint:      {QUANTIZED_ENDPOINT}")
print(f"Full-precision endpoint: {FULL_PRECISION_ENDPOINT}")
print(f"Region:                  {AWS_REGION}")

: 

## Deployment Configuration Summary

This comparison follows the pattern described in the blog: **choose the artifact first, then the runtime, then the AWS service.** The quantized GGUF artifact maps to llama.cpp in a custom container; the full-precision safetensor artifact maps to vLLM via SageMaker LMI.

In [ ]:
config_data = {
    "Property": [
        "HuggingFace Model ID",
        "Quantization",
        "Inference Runtime",
        "SageMaker Instance Type",
        "GPU",
        "Model Size on Disk",
        "Model Family",
    ],
    "Quantized Model": [
        "unsloth/Qwen3-VL-8B-Instruct-GGUF",
        "Unsloth Dynamic 2.0 \u2014 4-bit GGUF (Q4_K_M)",
        "llama.cpp (BYOC)",
        "ml.g5.xlarge",
        "NVIDIA A10G (24 GB)",
        "~6.4 GB (LLM + vision projector)",
        "Qwen3-VL (Vision-Language)",
    ],
    "Full-Precision Model": [
        "Qwen/Qwen3-VL-8B-Instruct",
        "None (BF16 full precision)",
        "vLLM (SageMaker LMI)",
        "ml.g5.12xlarge",
        "NVIDIA A10G x4 (96 GB)",
        "~16.4 GB",
        "Qwen3-VL (Vision-Language)",
    ],
}

df_config = pd.DataFrame(config_data)
display(HTML(
    "<p><em>Both models are Qwen3-VL (Vision-Language) variants that natively support "
    "image understanding. No different model family is required for image tasks.</em></p>"
    + df_config.to_html(index=False)
))

## Image-Based Comparisons

Each cell below sends the same image and prompt to both endpoints, displays the input image, shows the model outputs side by side, and plots per-prompt latency and throughput.

In [ ]:
def display_comparison(result: ComparisonResult):
    """Display side-by-side comparison of model outputs with metrics charts."""
    # Display the prompt
    display(HTML(f"<h3>Prompt: {result.prompt_text}</h3>"))
    
    # Display the input image if this is an image prompt
    if result.image_source:
        display(HTML("<h4>Input Image:</h4>"))
        if result.image_source.startswith("http"):
            display(IPImage(url=result.image_source, width=400))
        else:
            display(IPImage(filename=result.image_source, width=400))
    
    # Side-by-side outputs
    q = result.quantized
    fp = result.full_precision
    
    # Convert markdown formatting to HTML for table rendering
    import re
    def md_to_html(text):
        text = text.replace('\n', '<br>')
        text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
        text = re.sub(r'\*(.+?)\*', r'<em>\1</em>', text)
        text = re.sub(r'- ', r'&bull; ', text)
        return text
    
    q_text = q.error if q.error else md_to_html(q.generated_text)
    fp_text = fp.error if fp.error else md_to_html(fp.generated_text)
    
    html = f"""
    <table style="width:100%; border-collapse:collapse; margin:10px 0;">
    <tr style="background:#f0f0f0;">
        <th style="padding:10px; border:1px solid #ddd; width:50%;">{q.model_label}</th>
        <th style="padding:10px; border:1px solid #ddd; width:50%;">{fp.model_label}</th>
    </tr>
    <tr>
        <td style="padding:10px; border:1px solid #ddd; vertical-align:top; font-size:14px;">
            {q_text}
        </td>
        <td style="padding:10px; border:1px solid #ddd; vertical-align:top; font-size:14px;">
            {fp_text}
        </td>
    </tr>
    </table>
    """
    display(HTML(html))
    
    # Skip charts if both errored
    if q.error and fp.error:
        return
    
    # Per-prompt latency and throughput bar charts
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    labels = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
    colors = ["#2196F3", "#FF9800"]
    
    # Latency chart
    latencies = [q.latency_ms, fp.latency_ms]
    axes[0].bar(labels, latencies, color=colors)
    axes[0].set_ylabel("Latency (ms)")
    axes[0].set_title("End-to-End Latency")
    for i, v in enumerate(latencies):
        axes[0].text(i, v + max(latencies) * 0.02, f"{v:.0f}ms", ha="center", fontsize=10)
    
    # Throughput chart
    throughputs = [q.throughput_tps, fp.throughput_tps]
    axes[1].bar(labels, throughputs, color=colors)
    axes[1].set_ylabel("Tokens/sec")
    axes[1].set_title("Token Generation Throughput")
    for i, v in enumerate(throughputs):
        axes[1].text(i, v + max(throughputs) * 0.02, f"{v:.1f}", ha="center", fontsize=10)
    
    plt.tight_layout()
    plt.show()

### 1. Image Description

In [ ]:
# Image Description — Describe the contents of an image in detail
IMAGE_1 = "test_images/market.jpg"
PROMPT_1 = "Describe this image in one sentence."

result_1 = run_comparison(PROMPT_1, IMAGE_1, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_1)
display_comparison(result_1)

### 2. Object Identification

In [ ]:
# Object Identification — Identify and list objects in an image
IMAGE_2 = "test_images/market_singapore.jpg"
PROMPT_2 = "List the main objects in this image."

result_2 = run_comparison(PROMPT_2, IMAGE_2, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_2)
display_comparison(result_2)

### 3. OCR (Text Extraction)

In [ ]:
# OCR — Extract text from an image containing text
IMAGE_3 = "test_images/hollywood_sign.jpg"
PROMPT_3 = "What text is visible in this image?"

result_3 = run_comparison(PROMPT_3, IMAGE_3, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_3)
display_comparison(result_3)

### 4. Visual Question Answering

In [ ]:
# Visual QA — Answer a specific question about an image
IMAGE_4 = "test_images/cat_snow.jpg"
PROMPT_4 = "What is the animal doing and where is it?"

result_4 = run_comparison(PROMPT_4, IMAGE_4, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_4)
display_comparison(result_4)

### 5. Scene Understanding

In [ ]:
# Scene Understanding — Describe spatial relationships and context
IMAGE_5 = "test_images/shibuya_crossing.jpg"
PROMPT_5 = "Describe the layout of this scene in one sentence."

result_5 = run_comparison(PROMPT_5, IMAGE_5, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_5)
display_comparison(result_5)

## Text-Only Comparison

A single text-only prompt to compare non-visual performance as a secondary baseline.

### 6. Text Generation (No Image)

In [ ]:
# Text-only comparison — no image input
PROMPT_TEXT = "Explain the key benefits of model quantization for production ML deployments."

result_text = run_comparison(PROMPT_TEXT, None, GENERATION_PARAMS, CONFIG)
metrics_store.append(result_text)
display_comparison(result_text)

## Benchmark Evaluation

This section runs an objective quality evaluation using a structured benchmark dataset. Each benchmark entry has an image, a prompt, and an expected answer. Both models are evaluated against the same entries, and quality metrics (exact match, BLEU, ROUGE-L) are computed to quantify any degradation from quantization.

You can provide your own benchmark dataset (CSV or JSON) or use the built-in default dataset. See the README for dataset format details.

In [ ]:
# Install benchmark dependencies (nltk, rouge-score) if not already present
import subprocess, sys
for pkg in ["nltk", "rouge-score"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

# Download NLTK tokenizer data
import nltk
nltk.download("punkt_tab", quiet=True)

In [ ]:
# ── Benchmark Configuration ───────────────────────────────────────────────────────────────
# Set to a file path (CSV or JSON) to use a custom benchmark dataset,
# or leave as None to use the built-in default dataset.
BENCHMARK_DATASET_PATH = None  # e.g., "my_benchmark.csv"

In [ ]:
# Run benchmark evaluation against both endpoints
from benchmark_runner import (
    load_benchmark_dataset,
    run_benchmark,
    compute_degradation_report,
)

benchmark_dataset = load_benchmark_dataset(BENCHMARK_DATASET_PATH)
print(f"Loaded {len(benchmark_dataset)} benchmark entries")

benchmark_results = run_benchmark(benchmark_dataset, CONFIG, GENERATION_PARAMS)
degradation_report = compute_degradation_report(benchmark_results)

# Count successful vs failed entries
successful = sum(1 for r in benchmark_results if r.error is None)
failed = sum(1 for r in benchmark_results if r.error is not None)
print(f"Evaluated: {successful} successful, {failed} failed")

### Per-Entry Results

Detailed results for each benchmark entry showing the expected answer, both model outputs, and quality metrics.

In [ ]:
# Per-entry benchmark results table
import pandas as pd

rows = []
for r in benchmark_results:
    status = "✅" if r.error is None else f"❌ {r.error}"
    rows.append({
        "Image": r.entry.image_path.split("/")[-1] if "/" in r.entry.image_path else r.entry.image_path,
        "Category": r.entry.category or "—",
        "Prompt": r.entry.prompt[:50] + "..." if len(r.entry.prompt) > 50 else r.entry.prompt,
        "Expected": r.entry.expected_answer[:40] + "..." if len(r.entry.expected_answer) > 40 else r.entry.expected_answer,
        "Quantized": (r.quantized_answer[:40] + "...") if len(r.quantized_answer) > 40 else r.quantized_answer,
        "Full Precision": (r.full_precision_answer[:40] + "...") if len(r.full_precision_answer) > 40 else r.full_precision_answer,
        "Q-EM": f"{r.quantized_metrics.get('exact_match', 0):.0f}",
        "FP-EM": f"{r.full_precision_metrics.get('exact_match', 0):.0f}",
        "Q-BLEU": f"{r.quantized_metrics.get('bleu', 0):.2f}",
        "FP-BLEU": f"{r.full_precision_metrics.get('bleu', 0):.2f}",
        "Q-ROUGE": f"{r.quantized_metrics.get('rouge_l', 0):.2f}",
        "FP-ROUGE": f"{r.full_precision_metrics.get('rouge_l', 0):.2f}",
        "Status": status,
    })

df_benchmark = pd.DataFrame(rows)
display(HTML(df_benchmark.to_html(index=False)))

### Quality Metrics Summary

Aggregate quality scores comparing the quantized and full-precision models across all benchmark entries.

In [ ]:
# Benchmark quality metrics summary
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ["exact_match", "bleu", "rouge_l"]
metric_labels = ["Exact Match", "BLEU", "ROUGE-L"]
x = range(len(metrics))
width = 0.35

q_scores = [degradation_report.quantized_scores.get(m, 0) for m in metrics]
fp_scores = [degradation_report.full_precision_scores.get(m, 0) for m in metrics]

bars1 = ax.bar([i - width/2 for i in x], q_scores, width, label="Quantized (llama.cpp)", color="#2196F3")
bars2 = ax.bar([i + width/2 for i in x], fp_scores, width, label="Full Precision (vLLM)", color="#FF9800")

ax.set_ylabel("Score")
ax.set_title("Benchmark Quality Metrics — Quantized vs Full Precision")
ax.set_xticks(list(x))
ax.set_xticklabels(metric_labels)

# Dynamic y-axis: scale to data with 35% headroom for labels
max_val = max(max(q_scores, default=0), max(fp_scores, default=0))
y_top = max(max_val * 1.35, 0.05)  # at least 0.05 so empty charts look ok
ax.set_ylim(0, y_top)
label_offset = y_top * 0.02

ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + label_offset, f"{bar.get_height():.2f}", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + label_offset, f"{bar.get_height():.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

# Degradation report table
report_rows = []
for m, label in zip(metrics, metric_labels):
    q_val = degradation_report.quantized_scores.get(m, 0)
    fp_val = degradation_report.full_precision_scores.get(m, 0)
    abs_diff = degradation_report.absolute_diff.get(m, 0)
    rel_pct = degradation_report.relative_pct_change.get(m, 0)

    if q_val == fp_val:
        assessment = "No degradation"
    elif abs_diff > 0:
        assessment = f"Quantized +{abs_diff:.3f}"
    else:
        assessment = f"Quantized {abs_diff:.3f}"

    rel_str = f"{rel_pct:.1f}%" if rel_pct != float("inf") else "N/A"

    report_rows.append({
        "Metric": label,
        "Quantized": f"{q_val:.3f}",
        "Full Precision": f"{fp_val:.3f}",
        "Difference": f"{abs_diff:+.3f}",
        "Relative Change": rel_str,
        "Assessment": assessment,
    })

df_report = pd.DataFrame(report_rows)
display(HTML(df_report.to_html(index=False)))

In [ ]:
# Per-category breakdown (if categories are present in the dataset)
if degradation_report.per_category:
    categories = sorted(degradation_report.per_category.keys())
    
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 5))
    if len(metrics) == 1:
        axes = [axes]
    
    for idx, (m, label) in enumerate(zip(metrics, metric_labels)):
        q_vals = [degradation_report.per_category[c]["quantized_scores"][m] for c in categories]
        fp_vals = [degradation_report.per_category[c]["full_precision_scores"][m] for c in categories]
        
        x_cat = range(len(categories))
        axes[idx].bar([i - width/2 for i in x_cat], q_vals, width, label="Quantized", color="#2196F3")
        axes[idx].bar([i + width/2 for i in x_cat], fp_vals, width, label="Full Precision", color="#FF9800")
        axes[idx].set_ylabel("Score")
        axes[idx].set_title(f"{label} by Category")
        axes[idx].set_xticks(list(x_cat))
        axes[idx].set_xticklabels(categories, rotation=45, ha="right", fontsize=8)
        
        # Dynamic y-axis: scale to data with 35% headroom for labels
        cat_max = max(max(q_vals, default=0), max(fp_vals, default=0))
        cat_top = max(cat_max * 1.35, 0.05)
        axes[idx].set_ylim(0, cat_top)
        
        axes[idx].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print("No category breakdowns available (dataset has no category column).")

## Metrics Summary

Aggregated performance metrics across all prompts.

In [ ]:
# Average Latency — Overall and by prompt type
q_avg_lat, fp_avg_lat = compute_average_metrics(metrics_store, "latency_ms")
grouped = compute_grouped_averages(metrics_store)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#2196F3", "#FF9800"]

# Overall average latency
labels_overall = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
vals_overall = [q_avg_lat, fp_avg_lat]
axes[0].bar(labels_overall, vals_overall, color=colors)
axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("Average Latency — All Prompts")
for i, v in enumerate(vals_overall):
    axes[0].text(i, v + max(vals_overall) * 0.02, f"{v:.0f}ms", ha="center", fontsize=10)

# Latency by prompt type (grouped bar chart)
prompt_types = sorted(grouped.keys())
x = range(len(prompt_types))
width = 0.35
q_lats = [grouped[pt]["quantized_avg_latency_ms"] for pt in prompt_types]
fp_lats = [grouped[pt]["full_precision_avg_latency_ms"] for pt in prompt_types]

bars1 = axes[1].bar([i - width/2 for i in x], q_lats, width, label="Quantized", color=colors[0])
bars2 = axes[1].bar([i + width/2 for i in x], fp_lats, width, label="Full Precision", color=colors[1])
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Average Latency — By Prompt Type")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([pt.capitalize() for pt in prompt_types])
axes[1].legend()

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{bar.get_height():.0f}", ha="center", fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{bar.get_height():.0f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Average Throughput
q_avg_thr, fp_avg_thr = compute_average_metrics(metrics_store, "throughput_tps")

fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Quantized\n(llama.cpp)", "Full Precision\n(vLLM)"]
vals = [q_avg_thr, fp_avg_thr]
colors = ["#2196F3", "#FF9800"]

ax.bar(labels, vals, color=colors)
ax.set_ylabel("Tokens/sec")
ax.set_title("Average Token Generation Throughput — All Prompts")
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.02, f"{v:.1f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

### Cost Comparison

### Interpreting the Results

**On latency:** If the quantized model sometimes shows higher latency than the full-precision model, this reflects the difference in **serving stacks**, not just quantization. The full-precision endpoint runs on vLLM (a purpose-built GPU inference engine with CUDA-optimized kernels and tensor parallelism across 4x A10G GPUs), while the quantized endpoint runs on llama.cpp (a lightweight, portable engine on a single A10G GPU behind an nginx proxy). If both models ran on the same runtime, the quantized model would likely be faster.

**On throughput:** Similar token generation throughput is expected. The 4-bit model moves less data per token, but llama.cpp’s CUDA kernels aren’t as optimized as vLLM’s. These factors roughly cancel out.

**The real takeaway is cost-efficiency.** The quantized model delivers comparable output quality at **~`$1.41`/hr vs ~`$7.09`/hr** — a 5x cost reduction. For workloads where the quality difference is acceptable, quantization lets you serve the same model on dramatically cheaper infrastructure. That’s the core value proposition of Unsloth Dynamic 2.0: not just smaller models, but smaller bills.

In [ ]:
# Cost Comparison Summary Table
# Fetch live pricing from AWS Pricing API (falls back to hardcoded values)
live_pricing = get_pricing(AWS_REGION)
q_hourly = live_pricing["ml.g5.xlarge"]
fp_hourly = live_pricing["ml.g5.12xlarge"]
print(f"Pricing (source: {'live API' if q_hourly != PRICING['ml.g5.xlarge'] else 'hardcoded fallback'}):")
print(f"  ml.g5.xlarge:   ${q_hourly:.2f}/hr")
print(f"  ml.g5.12xlarge: ${fp_hourly:.2f}/hr")

q_cost_per_req = calculate_cost_per_request(q_avg_lat, q_hourly)
fp_cost_per_req = calculate_cost_per_request(fp_avg_lat, fp_hourly)

summary_data = {
    "Metric": [
        "Model",
        "Instance Type",
        "Hourly Cost (USD)",
        "Avg Latency (ms)",
        "Avg Throughput (tok/s)",
        "Est. Cost per Request (USD)",
        "Est. Cost per 1K Requests (USD)",
    ],
    "Quantized": [
        "Qwen3-VL-8B \u2014 4-bit GGUF (Q4_K_M)",
        "ml.g5.xlarge",
        f"${q_hourly:.2f}",
        f"{q_avg_lat:.0f}",
        f"{q_avg_thr:.1f}",
        f"${q_cost_per_req:.6f}",
        f"${q_cost_per_req * 1000:.4f}",
    ],
    "Full Precision": [
        "Qwen3-VL-8B \u2014 BF16",
        "ml.g5.12xlarge",
        f"${fp_hourly:.2f}",
        f"{fp_avg_lat:.0f}",
        f"{fp_avg_thr:.1f}",
        f"${fp_cost_per_req:.6f}",
        f"${fp_cost_per_req * 1000:.4f}",
    ],
}

df_summary = pd.DataFrame(summary_data)
display(HTML(df_summary.to_html(index=False)))

## Cleanup

Delete both SageMaker endpoints and their associated resources to stop incurring costs. If any deletion fails, run `terraform destroy` as a fallback.

In [ ]:
# Cleanup — Delete SageMaker endpoints, endpoint configurations, and models
confirm = input("\u26a0\ufe0f  This will DELETE both SageMaker endpoints. Type 'yes' to confirm: ")
if confirm.strip().lower() != 'yes':
    print("Cleanup skipped. Endpoints are still running.")
    raise SystemExit("Cleanup cancelled by user")

sm_client = boto3.client("sagemaker", region_name=AWS_REGION)

resources_to_delete = [
    ("endpoint", QUANTIZED_ENDPOINT),
    ("endpoint", FULL_PRECISION_ENDPOINT),
    ("endpoint_config", "qwen3-vl-8b-quantized-config"),
    ("endpoint_config", "qwen3-vl-8b-full-precision-config"),
    ("model", "qwen3-vl-8b-quantized"),
    ("model", "qwen3-vl-8b-full-precision"),
]

for resource_type, resource_name in resources_to_delete:
    try:
        if resource_type == "endpoint":
            sm_client.delete_endpoint(EndpointName=resource_name)
        elif resource_type == "endpoint_config":
            sm_client.delete_endpoint_config(EndpointConfigName=resource_name)
        elif resource_type == "model":
            sm_client.delete_model(ModelName=resource_name)
        print(f"✅ Deleted {resource_type}: {resource_name}")
    except sm_client.exceptions.ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "ValidationException" and "Could not find" in str(e):
            print(f"ℹ️  {resource_type} '{resource_name}' already deleted.")
        else:
            print(f"❌ Failed to delete {resource_type} '{resource_name}': {e}")
            print("   Fallback: run 'terraform destroy' from the terraform/ directory.")
    except Exception as e:
        print(f"❌ Failed to delete {resource_type} '{resource_name}': {e}")
        print("   Fallback: run 'terraform destroy' from the terraform/ directory.")

print("\nCleanup complete. If any deletions failed, run 'terraform destroy' as a fallback.")